# 🧪 Agent Evaluation — Financial Services Agents

Streamlined evaluation for LLM-only agents (no tools/ACR).

Tests:
1. Agent registration in Foundry
2. Router picks the correct specialist agent
3. End-to-end response quality
4. Cross-agent handoff detection
5. Summary report

> Update the configuration in Section 0 before running all cells.

## 0️⃣ Configuration

In [ ]:
import os

MULTI_AGENT_CONFIG = {
    "project_endpoint": os.environ["AZURE_AI_ENDPOINT"],
    "subscription_id": os.environ["AZURE_SUBSCRIPTION_ID"],
    "model_deployment": "gpt-4o",
    "agents": [
        {
            "name": "agent-summarizer",
            "description": "Summarizes financial services documents, extracting key findings, figures, and conclusions.",
            "version": "latest",
        },
        {
            "name": "User-clarification-agent",
            "description": "Asks follow-up questions to clarify ambiguous user requests about financial documents before other agents act.",
            "version": "latest",
        },
        {
            "name": "report-generator-agent",
            "description": "Generates structured financial reports, tables, and formatted output from analyzed document data.",
            "version": "latest",
        },
    ],
    "routing_tests": [
        ("Summarize the key points from this quarterly earnings report.", "agent-summarizer"),
        ("I'm not sure which document to look at — can you help me figure out what I need?", "User-clarification-agent"),
        ("Generate a formatted report comparing Q1 and Q2 revenue figures.", "report-generator-agent"),
        ("Give me a brief overview of the main takeaways from the risk assessment.", "agent-summarizer"),
        ("What do you mean by 'portfolio exposure'? Can you ask me more about what I need?", "User-clarification-agent"),
        ("Create a table showing year-over-year performance metrics.", "report-generator-agent"),
    ],
    "e2e_scenarios": [
        {
            "name": "Document summarization",
            "prompt": "Summarize the following financial document excerpt: 'Q3 revenue increased 12% YoY to $4.2B driven by strong demand in cloud services. Operating margins improved 200bps to 34%. Management raised full-year guidance.'",
            "expected_agent": "agent-summarizer",
            "validation_keywords": ["revenue", "cloud", "guidance"],
        },
        {
            "name": "Report generation",
            "prompt": "Generate a comparison report for: Q1 revenue $3.8B, Q2 revenue $4.0B, Q3 revenue $4.2B. Include growth rates.",
            "expected_agent": "report-generator-agent",
            "validation_keywords": ["Q1", "Q2", "Q3", "growth"],
        },
    ],
    "handoff_tests": [
        {
            "name": "Clarify then summarize",
            "prompt": "I have some financial documents but I'm not sure which ones are relevant. Once we figure that out, summarize the key findings.",
            "expected_primary_agent": "User-clarification-agent",
            "expected_secondary_agents": ["agent-summarizer"],
        },
        {
            "name": "Summarize then report",
            "prompt": "Analyze the key takeaways from our quarterly results and then produce a formatted executive summary report.",
            "expected_primary_agent": "agent-summarizer",
            "expected_secondary_agents": ["report-generator-agent"],
        },
    ],
}

print(f"Configured {len(MULTI_AGENT_CONFIG['agents'])} agents:")
for a in MULTI_AGENT_CONFIG["agents"]:
    print(f"  - {a['name']}")

## 1️⃣ Imports & Helpers

In [ ]:
import json
from datetime import datetime

from tabulate import tabulate
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
project_client = AIProjectClient(endpoint=MULTI_AGENT_CONFIG["project_endpoint"], credential=credential)
openai_client = project_client.get_openai_client()

results = []
agent_results = {a["name"]: [] for a in MULTI_AGENT_CONFIG["agents"]}
system_metrics = {
    "routing": {"passed": 0, "total": 0},
    "handoff": {"passed": 0, "total": 0},
}


def check(name, passed, detail=""):
    icon = "✅" if passed else "❌"
    results.append({"test": name, "passed": passed, "detail": detail, "section": name.split(":", 1)[0] if ":" in name else "system"})
    print(f"{icon} {name}" + (f" — {detail}" if detail else ""))
    return passed


def agent_check(agent_name, test_name, passed, detail=""):
    icon = "✅" if passed else "❌"
    record = {"agent": agent_name, "test": test_name, "passed": passed, "detail": detail,
              "section": test_name.split(":", 1)[0] if ":" in test_name else "other"}
    agent_results.setdefault(agent_name, []).append(record)
    print(f"{icon} [{agent_name}] {test_name}" + (f" — {detail}" if detail else ""))
    return passed


def model_to_dict(model):
    if model is None:
        return {}
    if isinstance(model, dict):
        return model
    if hasattr(model, "as_dict"):
        try:
            return model.as_dict()
        except Exception:
            pass
    return getattr(model, "_data", {}) if isinstance(getattr(model, "_data", None), dict) else {}


class AgentRouter:
    def __init__(self, agents, openai_client, model_name):
        self.agents = agents
        self.openai_client = openai_client
        self.model_name = model_name
        self.system_prompt = (
            "You are a specialist agent router. Always choose exactly one specialist agent as a tool call. "
            "Select the agent whose description most directly matches the user request."
        )

    def build_tools(self):
        return [
            {
                "type": "function",
                "function": {
                    "name": a["name"],
                    "description": a["description"],
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "reason": {"type": "string", "description": "Why this agent should handle the request."}
                        },
                        "required": [],
                    },
                },
            }
            for a in self.agents
        ]

    def route(self, prompt):
        response = self.openai_client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
            tools=self.build_tools(),
            tool_choice="auto",
        )
        tool_calls = getattr(response.choices[0].message, "tool_calls", None) or []
        selected = tool_calls[0].function.name if tool_calls else None
        return selected, tool_calls


def keyword_hits(text, keywords):
    lowered = (text or "").lower()
    return [kw for kw in keywords if kw.lower() in lowered]


router = AgentRouter(MULTI_AGENT_CONFIG["agents"], openai_client, MULTI_AGENT_CONFIG["model_deployment"])

print(f"Helpers loaded. Model: {MULTI_AGENT_CONFIG['model_deployment']}")

## 2️⃣ Agent Registration

Verify each agent is registered in the Foundry project.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: AGENT REGISTRATION")
print("=" * 70 + "\n")

for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]

    # Check agent exists in Foundry
    try:
        agent_obj = project_client.agents.get(agent_name=agent_name)
        agent_data = model_to_dict(agent_obj)
        agent_id = agent_data.get("id") or getattr(agent_obj, "id", None)
        agent_check(agent_name, "infra: Agent registered in Foundry", True, f"id={agent_id}")
    except Exception as exc:
        agent_check(agent_name, "infra: Agent registered in Foundry", False, str(exc)[:200])
        continue

    # Check version is active
    try:
        target_version = agent_cfg["version"]
        if target_version == "latest":
            versions = list(project_client.agents.list_versions(agent_name=agent_name))
            if versions:
                v_data = model_to_dict(versions[0])
                target_version = v_data.get("version") or getattr(versions[0], "version", "latest")
        version_obj = project_client.agents.get_version(agent_name=agent_name, agent_version=target_version)
        version_data = model_to_dict(version_obj)
        status = version_data.get("status") or version_data.get("state") or "unknown"
        agent_check(agent_name, "infra: Version active", str(status).lower() == "active",
                    f"version={target_version}; status={status}")
    except Exception as exc:
        agent_check(agent_name, "infra: Version active", False, str(exc)[:200])

## 3️⃣ Router / Orchestrator Test

Verify the LLM selects the correct specialist agent for each prompt.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: ROUTER / ORCHESTRATOR")
print("=" * 70 + "\n")

routing_total = len(MULTI_AGENT_CONFIG["routing_tests"])
routing_passed = 0

for prompt, expected_agent in MULTI_AGENT_CONFIG["routing_tests"]:
    selected_agent, tool_calls = router.route(prompt)
    passed = selected_agent == expected_agent
    routing_passed += int(passed)
    system_metrics["routing"]["total"] += 1
    system_metrics["routing"]["passed"] += int(passed)
    detail = f"expected={expected_agent}; actual={selected_agent}"
    check(f"routing: {prompt[:70]}", passed, detail)

routing_accuracy = (routing_passed / routing_total) if routing_total else 0.0
print(f"\nRouting accuracy: {routing_passed}/{routing_total} = {routing_accuracy:.1%}")

## 4️⃣ End-to-End Response Quality

Route the prompt to the correct agent, then have that agent generate a response and validate it contains expected keywords.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: END-TO-END RESPONSE QUALITY")
print("=" * 70 + "\n")


for scenario in MULTI_AGENT_CONFIG["e2e_scenarios"]:
    scenario_name = scenario["name"]
    prompt = scenario["prompt"]
    expected_agent = scenario["expected_agent"]
    validation_keywords = scenario["validation_keywords"]

    # Step 1: Route
    selected_agent, _ = router.route(prompt)
    if selected_agent != expected_agent:
        detail = f"Router mismatch: expected={expected_agent}; actual={selected_agent}"
        check(f"e2e: {scenario_name}", False, detail)
        continue

    # Step 2: Invoke the actual Foundry agent via responses API
    response = openai_client.responses.create(
        extra_body={
            "agent_reference": {
                "name": selected_agent,
                "type": "agent_reference",
            }
        },
        input=prompt,
    )
    final_text = response.output_text or ""

    # Step 3: Validate keywords
    matched = keyword_hits(final_text, validation_keywords)
    passed = len(matched) == len(validation_keywords)
    detail = (
        f"agent={selected_agent}; keywords={len(matched)}/{len(validation_keywords)}; "
        f"response_preview={final_text[:150]}"
    )
    check(f"e2e: {scenario_name}", passed, detail)
    agent_check(expected_agent, f"e2e: {scenario_name}", passed, detail)

## 5️⃣ Cross-Agent Handoff Tests

Test whether the system recognizes prompts needing multiple agents.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: CROSS-AGENT HANDOFF")
print("=" * 70 + "\n")

available_agents = [{"name": a["name"], "description": a["description"]} for a in MULTI_AGENT_CONFIG["agents"]]

handoff_system_prompt = (
    "You are a multi-agent orchestration planner. "
    "Given a user prompt and the available specialist agents, decide whether the work should be handled by one or multiple agents. "
    "Return JSON only with keys: needs_handoff (boolean), primary_agent (string), secondary_agents (array of strings), rationale (string)."
)

for test in MULTI_AGENT_CONFIG["handoff_tests"]:
    response = openai_client.chat.completions.create(
        model=MULTI_AGENT_CONFIG["model_deployment"],
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": handoff_system_prompt + " Available agents: " + json.dumps(available_agents),
            },
            {"role": "user", "content": test["prompt"]},
        ],
    )
    content = response.choices[0].message.content or "{}"
    try:
        plan = json.loads(content)
    except json.JSONDecodeError:
        plan = {"needs_handoff": False, "primary_agent": None, "secondary_agents": [], "rationale": content}

    expected_primary = test["expected_primary_agent"]
    expected_secondary = set(test.get("expected_secondary_agents", []))
    actual_primary = plan.get("primary_agent")
    actual_secondary = set(plan.get("secondary_agents", []))
    needs_handoff = bool(plan.get("needs_handoff"))
    passed = needs_handoff and actual_primary == expected_primary and expected_secondary.issubset(actual_secondary)

    system_metrics["handoff"]["total"] += 1
    system_metrics["handoff"]["passed"] += int(passed)
    detail = (
        f"expected_primary={expected_primary}; actual_primary={actual_primary}; "
        f"expected_secondary={sorted(expected_secondary)}; actual_secondary={sorted(actual_secondary)}; "
        f"rationale={plan.get('rationale', '')[:100]}"
    )
    check(f"handoff: {test['name']}", passed, detail)

## 6️⃣ Summary Report

In [ ]:
print("\n" + "═" * 80)
print("SUMMARY REPORT")
print("═" * 80)
print(f"Project endpoint: {MULTI_AGENT_CONFIG['project_endpoint']}")
print(f"Model deployment: {MULTI_AGENT_CONFIG['model_deployment']}")
print(f"Timestamp: {datetime.now().isoformat()}")
print("─" * 80)

all_agent_records = [r for records in agent_results.values() for r in records]
all_records = results + all_agent_records
total_passed = sum(1 for r in all_records if r["passed"])
total_count = len(all_records)
overall_score = (total_passed / total_count) if total_count else 0.0

print(f"System-wide score: {total_passed}/{total_count} = {overall_score:.1%}")
r = system_metrics["routing"]
h = system_metrics["handoff"]
print(f"Routing accuracy: {r['passed']}/{r['total']} = {(r['passed'] / r['total']) if r['total'] else 0:.1%}")
print(f"Handoff accuracy: {h['passed']}/{h['total']} = {(h['passed'] / h['total']) if h['total'] else 0:.1%}")
print()


def section_score(agent_name, section_prefix):
    recs = [r for r in agent_results.get(agent_name, []) if r["section"] == section_prefix]
    p = sum(1 for r in recs if r["passed"])
    return f"{p}/{len(recs)}" if recs else "—"


summary_rows = []
for a in MULTI_AGENT_CONFIG["agents"]:
    name = a["name"]
    summary_rows.append({
        "agent": name,
        "infra": section_score(name, "infra"),
        "e2e": section_score(name, "e2e"),
    })

print(tabulate(summary_rows, headers="keys", tablefmt="github"))

failing = [r for r in all_records if not r["passed"]]
if failing:
    print("\nFailures:")
    print(tabulate(
        [{"scope": r.get("agent", "system"), "test": r["test"], "detail": r.get("detail", "")} for r in failing],
        headers="keys", tablefmt="github",
    ))
else:
    print("\n🎉 No failures detected!")